# 01 · Análise exploratória das avaliações (Olist)

Objetivo: entender o conteúdo textual das avaliações para fundamentar as decisões de
**seleção**, **limpeza** e **tratamento de ausentes** da base de conhecimento.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 200)

In [ ]:
reviews = pd.read_csv(ROOT / "data/raw/olist_order_reviews_dataset.csv", encoding="utf-8")
print(reviews.shape)
reviews.head()

## Valores ausentes

In [ ]:
missing = reviews.isna().mean().sort_values(ascending=False).rename("% ausente") * 100
missing.round(1).to_frame()

## Notas: todas as avaliações vs. avaliações com comentário

In [ ]:
has_text = reviews["review_comment_message"].notna() | reviews["review_comment_title"].notna()
dist = pd.DataFrame({
    "todas": reviews["review_score"].value_counts().sort_index(),
    "com texto": reviews.loc[has_text, "review_score"].value_counts().sort_index(),
})
dist["% com texto"] = (dist["com texto"] / dist["todas"] * 100).round(1)
dist

In [ ]:
dist[["todas", "com texto"]].plot.bar(figsize=(8, 4), title="Distribuição de notas")
plt.xlabel("nota"); plt.ylabel("avaliações"); plt.tight_layout()

**Leitura:** clientes insatisfeitos comentam proporcionalmente muito mais
(~77% das notas 1 têm texto, contra ~36% das notas 5). A base de conhecimento terá,
portanto, mais peso de reclamações do que a base total — isso precisa ser dito ao
interpretar frequências.

## Tamanho dos textos

In [ ]:
lengths = reviews["review_comment_message"].dropna().str.len()
print(lengths.describe().round(1))
lengths.plot.hist(bins=50, figsize=(8, 3), title="Caracteres por comentário")

**Leitura:** textos muito curtos (mediana ~53 caracteres, máximo ~210). Não há necessidade de
*chunking*: cada avaliação vira um documento.

## Duplicidades

In [ ]:
dup_ids = reviews["review_id"].duplicated().sum()
print("review_id repetidos:", dup_ids)
reviews[reviews["review_id"].duplicated(keep=False)].sort_values("review_id").head(6)

In [ ]:
top_texts = reviews["review_comment_message"].dropna().str.strip().str.lower().value_counts().head(15)
top_texts

**Leitura:**
- o mesmo `review_id` aparece em pedidos diferentes (mesma avaliação replicada) → deduplicar por `review_id`;
- muitos textos idênticos e genéricos ("muito bom", "recebi o produto") → colapsar na **seleção de evidências**
  (não no índice), preservando a contagem de repetições.

## Amostras por nota

In [ ]:
for score in [1, 3, 5]:
    print(f"--- nota {score}")
    for t in reviews.query("review_score == @score")["review_comment_message"].dropna().sample(5, random_state=0):
        print(" •", t)